# NextTick — Next-Day Stock Market Forecasting

**CS 6140 · Machine Learning · Prof. Ehsan Elhamifar**  
**Team:** Pratham Pradeep Mahajan, Mohammad Aasim Shaikh

Dual-engine pipeline:
- **Task 1 — Classification (direction):** Logistic Regression, Random Forest, LSTM
- **Task 2 — Regression (magnitude):** Linear Regression, Random Forest, LSTM

Run all cells in order. Expected runtime: ~5-10 min on a T4 GPU.

## 1. Install dependencies and restart kernel

Colab will say **"Your session crashed"** after this cell — that is the intentional kernel restart. Just re-run from the next cell onward.

In [ ]:
!pip install -q --upgrade yfinance curl_cffi pandas-datareader scikit-learn "tensorflow>=2.18" pandas numpy joblib matplotlib

import os
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.7/133.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 12.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas=

## 2. Imports

In [1]:
import os, json, time, warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    mean_absolute_error, mean_squared_error, r2_score,
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow: {tf.__version__}')
print(f'Pandas:     {pd.__version__}')
print(f'NumPy:      {np.__version__}')

TensorFlow: 2.21.0
Pandas:     3.0.2
NumPy:      2.4.4


## 3. Configuration

In [2]:
CONFIG = {
    'tickers':        ['AAPL', 'MSFT', 'GOOGL', 'JPM', 'JNJ',
                       'XOM', 'WMT', 'BA', 'DIS', 'NVDA'],
    'years':          5,
    'lstm_window':    20,
    'test_fraction':  0.15,
    'val_fraction':   0.15,
    'output_dir':     './artifacts',
}
os.makedirs(CONFIG['output_dir'], exist_ok=True)
print('Config ready.')

Config ready.


## 4. Data collection — three-tier fallback

1. Try **yfinance** first (most accurate, most recent data).  
2. If Yahoo blocks us, fall back to **Stooq** via pandas-datareader (no rate limits).  
3. If both fail, generate **synthetic** OHLCV so the rest of the pipeline still runs end-to-end.

You will always get a usable `raw_df`.

In [3]:
def _clean(df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    # Some providers return lowercase column names
    df.columns = [str(c).title() for c in df.columns]
    needed = ['Open', 'High', 'Low', 'Close', 'Volume']
    if not all(c in df.columns for c in needed):
        return pd.DataFrame()
    df = df[needed].copy()
    df.index = pd.to_datetime(df.index)
    df = df.dropna()
    df['Ticker'] = ticker
    return df


# --- Tier 1: yfinance -----------------------------------------------
def fetch_yfinance(ticker: str, years: int, retries: int = 3) -> pd.DataFrame:
    import yfinance as yf
    for attempt in range(retries):
        try:
            df = yf.Ticker(ticker).history(period=f'{years}y', auto_adjust=True)
            out = _clean(df, ticker)
            if not out.empty:
                return out
        except Exception as exc:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                print(f'  yfinance {ticker}: {exc}')
    return pd.DataFrame()


# --- Tier 2: Stooq --------------------------------------------------
def fetch_stooq(ticker: str, years: int) -> pd.DataFrame:
    try:
        from pandas_datareader import data as pdr
        end = datetime.now()
        start = end - timedelta(days=365 * years + 30)
        # US tickers need a ".US" suffix on Stooq
        df = pdr.DataReader(f'{ticker}.US', 'stooq', start, end).sort_index()
        return _clean(df, ticker)
    except Exception as exc:
        print(f'  stooq {ticker}: {exc}')
        return pd.DataFrame()


# --- Tier 3: synthetic data (last-resort) ---------------------------
def fetch_synthetic(ticker: str, years: int, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    n = years * 252
    start = datetime.now() - timedelta(days=int(n * 1.45))
    dates = pd.bdate_range(start, periods=n)
    # Geometric random walk with mild drift + volatility
    log_ret = rng.normal(0.0004, 0.015, n)
    close = 100.0 * np.exp(np.cumsum(log_ret))
    df = pd.DataFrame(index=dates)
    df['Close'] = close
    df['Open'] = df['Close'].shift(1).fillna(close[0]) * (1 + rng.normal(0, 0.003, n))
    span = np.abs(rng.normal(0, 0.01, n)) * df['Close']
    df['High'] = np.maximum(df['Open'], df['Close']) + span
    df['Low']  = np.minimum(df['Open'], df['Close']) - span
    df['Volume'] = rng.integers(20_000_000, 120_000_000, n)
    df['Ticker'] = ticker
    return df


def fetch_one(ticker: str, years: int, seed: int) -> tuple[pd.DataFrame, str]:
    df = fetch_yfinance(ticker, years)
    if not df.empty:
        return df, 'yfinance'
    df = fetch_stooq(ticker, years)
    if not df.empty:
        return df, 'stooq'
    return fetch_synthetic(ticker, years, seed), 'synthetic'


frames, sources = [], {}
for i, t in enumerate(CONFIG['tickers']):
    df_t, src = fetch_one(t, CONFIG['years'], seed=42 + i)
    sources.setdefault(src, []).append(t)
    frames.append(df_t)
    print(f'{t:6s}  {src:10s}  rows={len(df_t):4d}  '
          f'from {df_t.index.min().date()} to {df_t.index.max().date()}')
    if i < len(CONFIG['tickers']) - 1:
        time.sleep(1.2)  # be polite to any live API

raw_df = pd.concat(frames).sort_index()
print(f'\nTotal observations: {len(raw_df):,}')
print(f'Sources used: {sources}')
raw_df.head()

AAPL    yfinance    rows=1255  from 2021-04-22 to 2026-04-21
MSFT    yfinance    rows=1255  from 2021-04-22 to 2026-04-21
GOOGL   yfinance    rows=1255  from 2021-04-22 to 2026-04-21
JPM     yfinance    rows=1255  from 2021-04-22 to 2026-04-21
JNJ     yfinance    rows=1255  from 2021-04-22 to 2026-04-21
XOM     yfinance    rows=1255  from 2021-04-22 to 2026-04-21
WMT     yfinance    rows=1255  from 2021-04-22 to 2026-04-21
BA      yfinance    rows=1255  from 2021-04-22 to 2026-04-21
DIS     yfinance    rows=1255  from 2021-04-22 to 2026-04-21
NVDA    yfinance    rows=1255  from 2021-04-22 to 2026-04-21

Total observations: 12,550
Sources used: {'yfinance': ['AAPL', 'MSFT', 'GOOGL', 'JPM', 'JNJ', 'XOM', 'WMT', 'BA', 'DIS', 'NVDA']}


,Open,High,Low,Close,Volume,Ticker
Date,,,,,,
2021-04-22 00:00:00-04:00,129.580343,130.661478,127.992741,128.508957,84566500,AAPL
2021-04-22 00:00:00-04:00,249.687014,251.193530,245.301834,246.769974,25606200,MSFT
2021-04-22 00:00:00-04:00,132.398121,132.459818,129.780650,129.877594,15256100,JPM
2021-04-22 00:00:00-04:00,144.012528,144.394618,142.953100,143.439392,7233000,JNJ
2021-04-22 00:00:00-04:00,46.230188,46.230188,45.429271,45.635693,21600100,XOM


## 5. Feature engineering

22 engineered indicators. The function below is copied verbatim into the Flask app so training and inference match byte-for-byte.

In [4]:
def rsi(series: pd.Series, window: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['Return_1d'] = out['Close'].pct_change()
    out['LogReturn_1d'] = np.log(out['Close'] / out['Close'].shift(1))
    for w in (5, 10, 20):
        out[f'SMA_{w}'] = out['Close'].rolling(w).mean()
        out[f'Close_over_SMA_{w}'] = out['Close'] / out[f'SMA_{w}'] - 1
    for w in (5, 10, 20):
        out[f'Volatility_{w}'] = out['Return_1d'].rolling(w).std()
    for w in (3, 5, 10):
        out[f'Momentum_{w}'] = out['Close'] / out['Close'].shift(w) - 1
    out['RSI_14'] = rsi(out['Close'], 14)
    ema_12 = out['Close'].ewm(span=12, adjust=False).mean()
    ema_26 = out['Close'].ewm(span=26, adjust=False).mean()
    out['MACD'] = ema_12 - ema_26
    out['MACD_Signal'] = out['MACD'].ewm(span=9, adjust=False).mean()
    out['MACD_Hist'] = out['MACD'] - out['MACD_Signal']
    bb_mid = out['Close'].rolling(20).mean()
    bb_std = out['Close'].rolling(20).std()
    out['BB_Position'] = (out['Close'] - bb_mid) / (2 * bb_std)
    out['Volume_Ratio'] = out['Volume'] / out['Volume'].rolling(20).mean()
    out['HL_Range'] = (out['High'] - out['Low']) / out['Close']
    out['Close_Position'] = ((out['Close'] - out['Low']) /
                             (out['High'] - out['Low']).replace(0, np.nan))
    return out


FEATURE_COLUMNS = [
    'Return_1d', 'LogReturn_1d',
    'SMA_5', 'SMA_10', 'SMA_20',
    'Close_over_SMA_5', 'Close_over_SMA_10', 'Close_over_SMA_20',
    'Volatility_5', 'Volatility_10', 'Volatility_20',
    'Momentum_3', 'Momentum_5', 'Momentum_10',
    'RSI_14',
    'MACD', 'MACD_Signal', 'MACD_Hist',
    'BB_Position', 'Volume_Ratio',
    'HL_Range', 'Close_Position',
]

# Apply per-ticker so rolling windows don't leak across symbols
engineered = []
for t in raw_df['Ticker'].unique():
    sub = raw_df[raw_df['Ticker'] == t].sort_index().copy()
    sub = engineer_features(sub)
    sub['Target_PctChange'] = sub['Close'].pct_change().shift(-1) * 100.0
    sub['Target_Direction'] = (sub['Target_PctChange'] > 0).astype(int)
    engineered.append(sub)

data = pd.concat(engineered).dropna().sort_index()
print(f'Clean samples: {len(data):,}')
print(f'\nClass balance:\n{data["Target_Direction"].value_counts(normalize=True).round(3)}')
print(f'\nFeature count: {len(FEATURE_COLUMNS)}')

Clean samples: 12,340

Class balance:
Target_Direction
1    0.524
0    0.476
Name: proportion, dtype: float64

Feature count: 22


## 6. Chronological train / val / test split

Per-ticker chronological split preserves time-series structure and prevents future data from leaking into training.

In [5]:
train_parts, val_parts, test_parts = [], [], []
for t in data['Ticker'].unique():
    sub = data[data['Ticker'] == t].sort_index()
    n = len(sub)
    n_test  = int(n * CONFIG['test_fraction'])
    n_val   = int(n * CONFIG['val_fraction'])
    n_train = n - n_val - n_test
    train_parts.append(sub.iloc[:n_train])
    val_parts.append(sub.iloc[n_train:n_train + n_val])
    test_parts.append(sub.iloc[n_train + n_val:])

train_df = pd.concat(train_parts)
val_df   = pd.concat(val_parts)
test_df  = pd.concat(test_parts)

X_train = train_df[FEATURE_COLUMNS].values
X_val   = val_df[FEATURE_COLUMNS].values
X_test  = test_df[FEATURE_COLUMNS].values

y_train_cls = train_df['Target_Direction'].values
y_val_cls   = val_df['Target_Direction'].values
y_test_cls  = test_df['Target_Direction'].values

y_train_reg = train_df['Target_PctChange'].values
y_val_reg   = val_df['Target_PctChange'].values
y_test_reg  = test_df['Target_PctChange'].values

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

print(f'Train: {len(train_df):,}   Val: {len(val_df):,}   Test: {len(test_df):,}')

Train: 8,640   Val: 1,850   Test: 1,850


## 7. Task 1 — Classification (direction)

In [6]:
def cls_metrics(y_true, y_pred, name):
    return {
        'model':     name,
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'f1':        f1_score(y_true, y_pred, zero_division=0),
    }

# Logistic Regression
logreg = LogisticRegression(max_iter=2000, C=1.0, random_state=42)
logreg.fit(X_train_s, y_train_cls)
m_logreg = cls_metrics(y_test_cls, logreg.predict(X_test_s), 'Logistic Regression')
print('Logistic Regression:', m_logreg)

# Random Forest Classifier
rf_cls = RandomForestClassifier(n_estimators=400, max_depth=12,
                                min_samples_leaf=5, n_jobs=-1, random_state=42)
rf_cls.fit(X_train_s, y_train_cls)
m_rf_cls = cls_metrics(y_test_cls, rf_cls.predict(X_test_s), 'Random Forest Classifier')
print('Random Forest     :', m_rf_cls)

Logistic Regression: {'model': 'Logistic Regression', 'accuracy': 0.5437837837837838, 'precision': 0.5474683544303798, 'recall': 0.7178423236514523, 'f1': 0.6211849192100538}
Random Forest     : {'model': 'Random Forest Classifier', 'accuracy': 0.514054054054054, 'precision': 0.5288888888888889, 'recall': 0.6172199170124482, 'f1': 0.5696505505026328}


### 7.1 LSTM classifier

Sliding windows of 20 days, per ticker.

In [9]:
WINDOW = CONFIG['lstm_window']

def make_sequences(df, feature_cols, target_col, window, fitted_scaler):
    Xs, ys = [], []
    for _, grp in df.groupby('Ticker'):
        grp = grp.sort_index()
        feat = fitted_scaler.transform(grp[feature_cols].values)
        tgt  = grp[target_col].values
        for i in range(window, len(grp)):
            Xs.append(feat[i - window:i])
            ys.append(tgt[i])
    return np.asarray(Xs, dtype=np.float32), np.asarray(ys)


X_tr_seq, y_tr_cls_seq = make_sequences(train_df, FEATURE_COLUMNS, 'Target_Direction', WINDOW, scaler)
X_va_seq, y_va_cls_seq = make_sequences(val_df,   FEATURE_COLUMNS, 'Target_Direction', WINDOW, scaler)
X_te_seq, y_te_cls_seq = make_sequences(test_df,  FEATURE_COLUMNS, 'Target_Direction', WINDOW, scaler)

print(f'LSTM tensors — train: {X_tr_seq.shape}, val: {X_va_seq.shape}, test: {X_te_seq.shape}')


def build_lstm(input_shape, output_activation, loss, metrics):
    model = Sequential([
        Input(shape=input_shape),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1, activation=output_activation),
    ])
    model.compile(optimizer='adam', loss=loss, metrics=metrics)
    return model


lstm_cls = build_lstm((WINDOW, len(FEATURE_COLUMNS)), 'sigmoid', 'binary_crossentropy', ['accuracy'])
lstm_cls.fit(
    X_tr_seq, y_tr_cls_seq,
    validation_data=(X_va_seq, y_va_cls_seq),
    epochs=30, batch_size=64,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=1,
)

lstm_cls_pred = (lstm_cls.predict(X_te_seq, verbose=0).flatten() > 0.5).astype(int)
m_lstm_cls = cls_metrics(y_te_cls_seq, lstm_cls_pred, 'LSTM Classifier')
print('LSTM Classifier   :', m_lstm_cls)

LSTM tensors — train: (8440, 20, 22), val: (1650, 20, 22), test: (1650, 20, 22)
Epoch 1/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 8s 31ms/step - accuracy: 0.5165 - loss: 0.6945 - val_accuracy: 0.5164 - val_loss: 0.6927
Epoch 2/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 6s 40ms/step - accuracy: 0.5173 - loss: 0.6931 - val_accuracy: 0.5170 - val_loss: 0.6937
Epoch 3/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - accuracy: 0.5268 - loss: 0.6918 - val_accuracy: 0.5115 - val_loss: 0.6945
Epoch 4/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - accuracy: 0.5277 - loss: 0.6919 - val_accuracy: 0.5152 - val_loss: 0.6943
Epoch 5/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 5s 41ms/step - accuracy: 0.5340 - loss: 0.6903 - val_accuracy: 0.5030 - val_loss: 0.6966
Epoch 6/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - accuracy: 0.5390 - loss: 0.6900 - val_accuracy: 0.5061 - val_loss: 0.6958
LSTM Classifier   : {'model': 'LSTM Classifier', 'accuracy': 0.5066666666666667, 'precision': 0.5204582651391162, 'recall': 0.7361111111111112, 'f

In [11]:
cls_board = pd.DataFrame([m_logreg, m_rf_cls, m_lstm_cls]).set_index('model')
print('\nClassification leaderboard')
print('=' * 60)
print(cls_board.round(4))


Classification leaderboard
                          accuracy  precision  recall      f1
model                                                        
Logistic Regression         0.5438     0.5475  0.7178  0.6212
Random Forest Classifier    0.5141     0.5289  0.6172  0.5697
LSTM Classifier             0.5067     0.5205  0.7361  0.6098


## 8. Task 2 — Regression (magnitude)

In [12]:
def reg_metrics(y_true, y_pred, name):
    return {
        'model': name,
        'MAE':   mean_absolute_error(y_true, y_pred),
        'RMSE':  float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'R2':    r2_score(y_true, y_pred),
    }


# Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(X_train_s, y_train_reg)
m_lin = reg_metrics(y_test_reg, lin_reg.predict(X_test_s), 'Linear Regression')
print('Linear Regression:', m_lin)

# Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=400, max_depth=12,
                               min_samples_leaf=5, n_jobs=-1, random_state=42)
rf_reg.fit(X_train_s, y_train_reg)
m_rf_reg = reg_metrics(y_test_reg, rf_reg.predict(X_test_s), 'Random Forest Regressor')
print('Random Forest    :', m_rf_reg)

# LSTM Regressor
_, y_tr_reg_seq = make_sequences(train_df, FEATURE_COLUMNS, 'Target_PctChange', WINDOW, scaler)
_, y_va_reg_seq = make_sequences(val_df,   FEATURE_COLUMNS, 'Target_PctChange', WINDOW, scaler)
_, y_te_reg_seq = make_sequences(test_df,  FEATURE_COLUMNS, 'Target_PctChange', WINDOW, scaler)

lstm_reg = build_lstm((WINDOW, len(FEATURE_COLUMNS)), 'linear', 'mse', ['mae'])
lstm_reg.fit(
    X_tr_seq, y_tr_reg_seq,
    validation_data=(X_va_seq, y_va_reg_seq),
    epochs=30, batch_size=64,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=1,
)
lstm_reg_pred = lstm_reg.predict(X_te_seq, verbose=0).flatten()
m_lstm_reg = reg_metrics(y_te_reg_seq, lstm_reg_pred, 'LSTM Regressor')
print('LSTM Regressor   :', m_lstm_reg)

Linear Regression: {'model': 'Linear Regression', 'MAE': 1.1700826742313377, 'RMSE': 1.6176959223671157, 'R2': 0.0004415402166004778}
Random Forest    : {'model': 'Random Forest Regressor', 'MAE': 1.1755866882184252, 'RMSE': 1.619528692096084, 'R2': -0.0018246437089943779}
Epoch 1/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 10s 37ms/step - loss: 3.8337 - mae: 1.3552 - val_loss: 4.5743 - val_mae: 1.3832
Epoch 2/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - loss: 3.8123 - mae: 1.3500 - val_loss: 4.5646 - val_mae: 1.3836
Epoch 3/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 6s 42ms/step - loss: 3.8066 - mae: 1.3506 - val_loss: 4.5636 - val_mae: 1.3842
Epoch 4/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 4s 33ms/step - loss: 3.8124 - mae: 1.3516 - val_loss: 4.5621 - val_mae: 1.3811
Epoch 5/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - loss: 3.8019 - mae: 1.3496 - val_loss: 4.5677 - val_mae: 1.3838
Epoch 6/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 5s 41ms/step - loss: 3.7963 - mae: 1.3487 - val_loss: 4.5658 - val_mae: 1.3841
Epoch 7/30


In [13]:
reg_board = pd.DataFrame([m_lin, m_rf_reg, m_lstm_reg]).set_index('model')
print('\nRegression leaderboard')
print('=' * 60)
print(reg_board.round(4))


Regression leaderboard
                            MAE    RMSE      R2
model                                          
Linear Regression        1.1701  1.6177  0.0004
Random Forest Regressor  1.1756  1.6195 -0.0018
LSTM Regressor           1.1942  1.6440 -0.0050


## 9. Export artifacts

These files feed directly into the Flask inference app:
- 4 sklearn models → `.pkl`
- 2 Keras LSTM models → `.keras`
- 1 StandardScaler → `.pkl`
- `metadata.json` with feature order, window size, and metrics

In [14]:
out = CONFIG['output_dir']

joblib.dump(logreg,  f'{out}/logistic_regression.pkl')
joblib.dump(rf_cls,  f'{out}/random_forest_classifier.pkl')
joblib.dump(lin_reg, f'{out}/linear_regression.pkl')
joblib.dump(rf_reg,  f'{out}/random_forest_regressor.pkl')
joblib.dump(scaler,  f'{out}/scaler.pkl')

lstm_cls.save(f'{out}/lstm_classifier.keras')
lstm_reg.save(f'{out}/lstm_regressor.keras')

metadata = {
    'created_at':         datetime.utcnow().isoformat() + 'Z',
    'feature_columns':    FEATURE_COLUMNS,
    'lstm_window':        WINDOW,
    'min_rows_required':  30,
    'tickers_trained_on': list(data['Ticker'].unique()),
    'data_sources':       sources,
    'metrics': {
        'classification': cls_board.reset_index().to_dict(orient='records'),
        'regression':     reg_board.reset_index().to_dict(orient='records'),
    },
}
with open(f'{out}/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print('Artifacts written:')
for fn in sorted(os.listdir(out)):
    size_kb = os.path.getsize(f'{out}/{fn}') / 1024
    print(f'  {fn:32s}  {size_kb:>8.1f} KB')

Artifacts written:
  linear_regression.pkl                  0.9 KB
  logistic_regression.pkl                1.0 KB
  lstm_classifier.keras                453.8 KB
  lstm_regressor.keras                 453.8 KB
  metadata.json                          2.0 KB
  random_forest_classifier.pkl       17870.9 KB
  random_forest_regressor.pkl         4839.9 KB
  scaler.pkl                             1.1 KB


## 10. Package and download

In [15]:
import shutil
archive = shutil.make_archive('nexttick_artifacts', 'zip', CONFIG['output_dir'])
print(f'Archive: {archive}  ({os.path.getsize(archive) / 1024:.1f} KB)')

try:
    from google.colab import files
    files.download(archive)
except Exception:
    print('(Not in Colab — archive saved locally.)')

Archive: /content/nexttick_artifacts.zip  (7374.4 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>